In [14]:
import time
import threading
from copy import deepcopy

import numpy as np
import pinocchio as pin
from pinocchio.shortcuts import buildModelsFromMJCF

import crocoddyl

from unitree_sdk2py.core.channel import (
    ChannelFactoryInitialize,
    ChannelPublisher,
    ChannelSubscriber,
)
from unitree_sdk2py.idl.default import unitree_hg_msg_dds__LowState_ as LowState_default
from unitree_sdk2py.idl.default import unitree_hg_msg_dds__LowCmd_ as LowCmd_default
from unitree_sdk2py.idl.unitree_hg.msg.dds_ import LowCmd_, LowState_

from robot_assets.robots.h12_constants import TOPIC_LOWCMD, TOPIC_LOWSTATE

from mpc_controller.mpc_builder import MPCController, MPCOCP
from scripts.build_squat_ocp import build_squat_ocp, build_squat_com_trajectory

from scripts.build_squat_ocp import *
from scripts.robot_loader import *

from scripts.debug_unitree_dds import print_low_state, print_low_cmd


xml_path = "/home/cpene/Documents/robot_playground/.venv/lib/python3.12/site-packages/robot_assets/models/h12/scene/h12_27dof.xml"
mesh_path = "/home/cpene/Documents/robot_playground/.venv/lib/python3.12/site-packages/robot_assets"

with open(xml_path, "r") as f:
    for _ in range(5):
        print(f.readline().strip())

model_pinocchio, collision_model, visual_model = buildModelsFromMJCF(
    xml_path,
    root_joint=pin.JointModelFreeFlyer(),
)
# change name of first link
try:
    jid = model_pinocchio.getJointId("floating_base_joint")
    model_pinocchio.names[jid] = "root_joint"
    for fid, frame in enumerate(model_pinocchio.frames):
        if frame.name == "floating_base_joint":
            model_pinocchio.frames[fid].name = "root_joint"
except Exception:
    pass
model_pinocchio = add_contact_frames(model_pinocchio, contact_z_offset=0.05)
data_pinocchio = model_pinocchio.createData()

# get ids for contacts
left_foot_id = model_pinocchio.getFrameId("left_ground")
right_foot_id = model_pinocchio.getFrameId("right_ground")
contact_frame_ids = [left_foot_id, right_foot_id]

# neutral configuration - I don't know if I will use it, better get it from DDS topics i guess
q0_pin = pin.neutral(model_pinocchio)
v0 = np.zeros(model_pinocchio.nv)
x0 = np.concatenate([q0_pin, v0])
full_gravity_torques = pin.rnea(model_pinocchio, data_pinocchio, q0_pin, v0, np.zeros_like(v0))
u0 = full_gravity_torques[6:]

print("u0 shape =", u0.shape)

<mujoco model="h1_2">
<compiler angle="radian" meshdir="meshes/" autolimits="true"/>
<statistic meansize="0.144785" extent="1.23314" center="0.025392 2.0634e-05 -0.245975"/>
<default>
<geom contype="1" conaffinity="1" solref="0.005 1" friction="1.0 0.005 0.0001"/>
Frame 'left_ground' added successfully.
Frame 'right_ground' added successfully.
u0 shape = (27,)


In [15]:
TRAJECTORY_DURATION = 3.0
HORIZON_LENGTH = 50
OCP_DT = 0.02          # 50 Hz

CHANNEL_ID = 79         # DDS channel
NET_INTERFACE = "lo"   # ex: "lo" pour loopback

reference_com_trajectory = build_squat_com_trajectory(
    model_pinocchio,
    data_pinocchio,
    q0_pin,
    v0,
    traj_duration=TRAJECTORY_DURATION,
    ocp_dt=OCP_DT,
    horizon_length=HORIZON_LENGTH,
    squat_depth=0.20,
)

ocp = build_squat_ocp(
    x0,
    model_pinocchio,
    data_pinocchio,
    reference_com_trajectory,
    contact_frame_ids,
    OCP_DT,
    HORIZON_LENGTH,
)

mpc = MPCController(
    ocp=ocp,
    x0=x0,
    u0=u0,
    max_iter=10,
    com_ref_traj=reference_com_trajectory,
    verbose=True,
)

print("MPC ready. dt =", mpc.dt, ", horizon length =", mpc.N)

MPC ready. dt = 0.02 , horizon length = 50


In [16]:
# prepare DDS subscription for MPC : subscriber LowState_, publisher LowCmd_

latest_low_state = None
low_state_lock = threading.Lock()

def low_state_callback(msg: LowState_):
    global latest_low_state
    with low_state_lock:
        latest_low_state = deepcopy(msg)

def init_dds():
    print("Initializing DDS...")
    ChannelFactoryInitialize(CHANNEL_ID, NET_INTERFACE)

    # Subscriber état
    sub_state = ChannelSubscriber(TOPIC_LOWSTATE, LowState_)
    sub_state.Init(low_state_callback, 10)

    # Publisher commande
    pub_cmd = ChannelPublisher(TOPIC_LOWCMD, LowCmd_)
    pub_cmd.Init()

    return pub_cmd

# LowState_ conversion for OCP -> x = [q, v]
def lowstate_to_x(low_state: LowState_, model: pin.Model) -> np.ndarray:
    """
    Construit l'état x = [q, v] pour l'OCP à partir du LowState DDS.

    Hypothèses :
      - base position = [0, 0, 0] (on travaille en base fixée en translation),
      - orientation base = quaternion IMU,
      - vitesse linéaire base = 0,
      - vitesse angulaire base = gyro IMU,
      - articulaires = motor_state[0..nu-1].
    """
    nq = model.nq
    nv = model.nv

    n_model_joints = nq - 7 # without flotting base

    n_motors_dds = len(low_state.motor_state) # number of motors in low_state message = 35

    n = min(n_model_joints, n_motors_dds)

    q = np.zeros(nq)
    v = np.zeros(nv)

    # Base translation (3) : on la fige à zéro
    q[0:3] = 0.0

    # Orientation : MuJoCo/IMU est typiquement [w, x, y, z]
    qw, qx, qy, qz = low_state.imu_state.quaternion
    # Pinocchio attend [x, y, z, w]
    q[3:7] = np.array([qx, qy, qz, qw])



    # Vitesse base :
    #   - linéaire : 0 (non mesurée)
    #   - angulaire : gyro IMU
    # v = [vx, vy, vz, wx, wy, wz, joint_velocities...]
    v[0:3] = 0.0
    v[3:6] = np.array(low_state.imu_state.gyroscope)


    # Joints q and v
    motor_q  = np.array([m.q  for m in low_state.motor_state[:n]], dtype=float)
    motor_dq = np.array([m.dq for m in low_state.motor_state[:n]], dtype=float)

    q[7 : 7 + n] = motor_q
    v[6 : 6 + n] = motor_dq
    
    x = np.concatenate([q, v])
    return x


def send_pd_hold(pub_cmd, ls, q_ref_stand, kp_pd=50.0, kd_pd=1.0):
    cmd = LowCmd_default()

    # modes globaux (optionnel, en simu c'est surtout pour être cohérent)
    cmd.mode_pr = ls.mode_pr
    cmd.mode_machine = ls.mode_machine

    n_motors_dds   = len(ls.motor_state)
    nq             = model_pinocchio.nq
    n_model_joints = nq - 7
    n              = min(n_motors_dds, n_model_joints)

    for i in range(n):
        motor_state = ls.motor_state[i]
        motor_cmd   = cmd.motor_cmd[i]

        q  = motor_state.q
        dq = motor_state.dq
        q_ref = q_ref_stand[i]

        tau_pd = kp_pd * (q_ref - q) - kd_pd * dq

        motor_cmd.mode = motor_state.mode
        motor_cmd.q    = q
        motor_cmd.dq   = dq
        motor_cmd.kp   = 0.0
        motor_cmd.kd   = 0.0
        motor_cmd.tau  = float(tau_pd)

    # les moteurs restants sont laissés neutres
    for i in range(n, n_motors_dds):
        motor_state = ls.motor_state[i]
        motor_cmd   = cmd.motor_cmd[i]
        motor_cmd.mode = motor_state.mode
        motor_cmd.q    = motor_state.q
        motor_cmd.dq   = motor_state.dq
        motor_cmd.kp   = 0.0
        motor_cmd.kd   = 0.0
        motor_cmd.tau  = 0.0

    pub_cmd.Write(cmd)

def debug_print_lowstate(ls, max_motors=6):
    print("=== LowState debug ===")

    # Modes globaux (s'ils existent)
    if hasattr(ls, "mode_pr"):
        print("mode_pr     :", ls.mode_pr)
    if hasattr(ls, "mode_machine"):
        print("mode_machine:", ls.mode_machine)

    # IMU
    if hasattr(ls, "imu_state"):
        imu = ls.imu_state
        if hasattr(imu, "quaternion"):
            print("IMU quaternion   :", [float(x) for x in imu.quaternion])
        if hasattr(imu, "gyroscope"):
            print("IMU gyroscope    :", [float(x) for x in imu.gyroscope])
        if hasattr(imu, "accelerometer"):
            print("IMU accelerom.   :", [float(x) for x in imu.accelerometer])

    # Quelques moteurs
    if hasattr(ls, "motor_state"):
        print(f"nb moteurs dans LowState: {len(ls.motor_state)}")
        for i, m in enumerate(ls.motor_state[:max_motors]):
            # adapte les champs selon le vrai MotorState_ (q, dq, tau_est ?)
            qs = getattr(m, "q", None)
            dqs = getattr(m, "dq", None)
            taus = getattr(m, "tau", None)  # ou tau_est selon l'IDL

            print(f"  motor[{i}] q={qs}, dq={dqs}, tau={taus}")
    print("=======================")

def debug_lowstate_and_pinocchio_joint_mapping():
    print("Waiting for first LowState message...")
    while True:
        with low_state_lock:
            ls = deepcopy(latest_low_state)
        if ls is not None:
            break
        time.sleep(0.001)
    print("First LowState received.")

    # 1) Debug brut LowState
    debug_print_lowstate(ls)

    # 2) Conversion vers x = [q; v]
    x = lowstate_to_x(ls, model_pinocchio)
    nq = model_pinocchio.nq
    nv = model_pinocchio.nv
    q = x[:nq]
    v = x[nq:nq+nv]

    print("nq, nv:", nq, nv)
    print("q[:10]:", q[:10])
    print("v[:10]:", v[:10])

    # 3) Comparaison joint par joint
    n_motors_dds   = len(ls.motor_state)
    n_model_joints = nq - 7           # base flottante 7 (3 pos + 4 quat)
    n              = min(n_motors_dds, n_model_joints)

    print("\n=== Comparaison joint DDS vs Pinocchio ===")
    for i in range(min(n, 10)):  # affiche les 10 premiers pour commencer
        q_dds = ls.motor_state[i].q
        q_pin = q[7 + i]  # si ton lowstate_to_x met la base en premier
        print(f"joint {i:2d}: DDS q={q_dds:+.4f},   Pinocchio q={q_pin:+.4f}")
    print("===========================================")


In [17]:
def run_mpc():
    pub_cmd = init_dds()

    # 1) Attendre la première mesure d'état
    print("Waiting for first LowState message...")
    while True:
        with low_state_lock:
            ls0 = latest_low_state
        if ls0 is not None:
            break
        time.sleep(0.001)
    print("First LowState received.")
    debug_print_lowstate(ls0)

    # 1b) Construire la référence de maintien
    n_motors_dds   = len(ls0.motor_state)
    nq             = model_pinocchio.nq
    n_model_joints = nq - 7
    n              = min(n_motors_dds, n_model_joints)
    q_ref_stand = np.array([m.q for m in ls0.motor_state[:n]], dtype=float)

    CONTROL_DT = mpc.dt
    T_total = TRAJECTORY_DURATION
    N_ctrl = int(T_total / CONTROL_DT)

    print(f"CONTROL_DT={CONTROL_DT}, N_ctrl={N_ctrl}")

    ocp_save = []

    for k in range(N_ctrl):
        t0 = time.time()

        # 2) Récupérer l'état courant DDS
        with low_state_lock:
            ls = deepcopy(latest_low_state)

        if ls is None:
            continue

        # 3) Conversion en état pour l’OCP
        x_meas = lowstate_to_x(ls, model_pinocchio)
        debug_print_lowstate(ls)

        if k == 0:
            # --- PREMIERE ITERATION : on envoie immédiatement un PD de maintien ---
            send_pd_hold(pub_cmd, ls, q_ref_stand)

            # On lance le MPC pour le "réchauffer", mais on ignore cette première commande
            dt_mpc, _ = mpc.step(x_meas)
            assert np.isclose(dt_mpc, CONTROL_DT)

        else:
            # --- ITERATIONS SUIVANTES : on utilise le MPC normalement ---
            dt_mpc, u_optimal = mpc.step(x_meas)
            assert np.isclose(dt_mpc, CONTROL_DT)


            # 5) Envoi commande via DDS (comme on a déjà fait)
            cmd = LowCmd_default()
            cmd.mode_pr = ls.mode_pr
            cmd.mode_machine = ls.mode_machine

            n_motors_dds   = len(ls.motor_state)
            nq             = model_pinocchio.nq
            n_model_joints = nq - 7
            n              = min(n_motors_dds, n_model_joints)

            assert u_optimal.shape[0] == n

            for i in range(n):
                motor_state = ls.motor_state[i]
                motor_cmd   = cmd.motor_cmd[i]

                motor_cmd.mode = motor_state.mode
                motor_cmd.q    = motor_state.q
                motor_cmd.dq   = motor_state.dq
                motor_cmd.kp   = 0.0
                motor_cmd.kd   = 0.0
                motor_cmd.tau  = float(u_optimal[i])

            for i in range(n, n_motors_dds):
                motor_state = ls.motor_state[i]
                motor_cmd   = cmd.motor_cmd[i]
                motor_cmd.mode = motor_state.mode
                motor_cmd.q    = motor_state.q
                motor_cmd.dq   = motor_state.dq
                motor_cmd.kp   = 0.0
                motor_cmd.kd   = 0.0
                motor_cmd.tau  = 0.0

            pub_cmd.Write(cmd)

        # 6) Temps réel
        elapsed = time.time() - t0
        to_sleep = CONTROL_DT - elapsed
        if to_sleep > 0:
            time.sleep(to_sleep)

    print("MPC loop finished.")
    return ocp_save

def send_pd_hold_current_pose(pub_cmd):
    # 1) lire le dernier LowState
    with low_state_lock:
        ls = deepcopy(latest_low_state)
    if ls is None:
        return

    # 2) créer une commande avec valeurs par défaut
    cmd = LowCmd_default()

    # gains PD
    KP = 40.0
    KD = 1.0

    # 3) on borne au plus petit nombre de moteurs entre état et commande
    n = min(len(ls.motor_state), len(cmd.motor_cmd))

    for i in range(n):
        ms = ls.motor_state[i]
        mc = cmd.motor_cmd[i]

        # consigne = position actuelle (on maintient)
        mc.q  = ms.q
        mc.dq = 0.0
        mc.tau = 0.0
        mc.kp = KP
        mc.kd = KD

    # 4) envoi DDS
    pub_cmd.Write(cmd)

In [ ]:
#init_dds()
#debug_lowstate_and_pinocchio_joint_mapping()
#ocp_log = run_mpc()


def watch_free_fall(duration=2.0, period=0.50, max_motors=12):
    """
    Print LowState_ (and LowCmd_ if available) regularly during a free-fall.

    - duration : total recording time in seconds
    - period   : time step between prints in seconds
    - max_motors : max number of motors to display in print_low_state/print_low_cmd
    """
    
    print("Waiting for first LowState message...")
    while True:
        with low_state_lock:
            ls0 = latest_low_state
        if ls0 is not None:
            break
        time.sleep(0.001)
    print("First LowState received.")
    debug_print_lowstate(ls0)
    
    t_start = time.time()
    print(f"Starting free-fall watch for {duration} s, period = {period} s")

    while True:
        t_now = time.time()
        if t_now - t_start > duration:
            break

        # Copy last LowState under lock
        with low_state_lock:
            ls = deepcopy(latest_low_state)

        if ls is not None:
            t_rel = t_now - t_start
            print(f"\n===== t = {t_rel: .3f} s =====")
            print_low_state(ls, max_motors=max_motors)

        time.sleep(period)

    print("Free-fall watch finished.")

import csv

def record_free_fall_csv(csv_path="free_fall_log.csv",
                         duration=2.0,
                         period=0.002,
                         joint_indices=(0, 1, 2)):
    """
    Record selected joint states during free-fall into a CSV file.

    - csv_path      : output CSV file path
    - duration      : total recording time [s]
    - period        : sampling period [s]
    - joint_indices : tuple of motor indices to log
    """
    print("Waiting for first LowState message...")
    while True:
        with low_state_lock:
            ls0 = latest_low_state
        if ls0 is not None:
            break
        time.sleep(0.001)
    print("First LowState received.")
    debug_print_lowstate(ls0)

    t_start = time.time()
    print(f"Recording free-fall to '{csv_path}' for {duration} s")

    with open(csv_path, "w", newline="") as f:
        writer = csv.writer(f)
        # Header
        header = ["t"]
        for j in joint_indices:
            header += [f"q_{j}", f"dq_{j}", f"ddq_{j}", f"tau_est_{j}"]
        writer.writerow(header)

        while True:
            t_now = time.time()
            if t_now - t_start > duration:
                break

            with low_state_lock:
                ls = deepcopy(latest_low_state)

            if ls is not None:
                row = [t_now - t_start]
                for j in joint_indices:
                    ms = ls.motor_state[j]
                    row += [ms.q, ms.dq, ms.ddq, ms.tau_est]
                writer.writerow(row)

            time.sleep(period)

    print("CSV recording finished.")


pub_cmd = init_dds()
record_free_fall_csv("free_fall.csv", duration=3.0, period=0.002, joint_indices=(0, 1, 2))


Initializing DDS...
Waiting for first LowState message...
